# Clean and export the QAChat repository

This notebook prepares a safe source snapshot of QAChat for review and
collaboration. It has two separate operations:

1. **Preview and optionally remove** recognized, disposable cache files and
   operating-system metadata. Deletion is **off by default**.
2. **Preview and optionally export** a clean ZIP containing the current
   working-tree contents of Git-tracked files, plus `.gitignore` and an
   explicitly empty `sandbox/` directory. Export is **off by default**.

Run the notebook from the QAChat checkout (normally in `assets/`). Git is
required for the export, but not for the limited cleanup. Untracked and
ignored files are never added automatically to the ZIP. Add any intended new
source files to Git first, e.g. `git add <path>`, then rerun the preview.

**Protected data:** Neither operation deletes or exports the contents of
`sandbox/`, local model caches, virtual environments, `.git/`, or the
benchmark workspaces. The exporter does **not** clean the working tree.


In [ ]:
from __future__ import annotations

import os
import shutil
import stat
import subprocess
import tempfile
import tomllib
import zipfile
from pathlib import Path


In [ ]:
def find_repository_root(start: Path | None = None) -> Path:
    """Find the QAChat checkout containing the expected project markers.

    Parameters
    ----------
    start : Path | None
        Starting location, or the current working directory.

    Returns
    -------
    Path
        Resolved QAChat repository root.

    Raises
    ------
    RuntimeError
        If the location is not inside a recognizable QAChat checkout.

    """

    current = (start or Path.cwd()).resolve()
    if current.is_file():
        current = current.parent

    for directory in (current, *current.parents):
        pyproject = directory / "pyproject.toml"
        if not (
            pyproject.is_file()
            and (directory / "qachat" / "__init__.py").is_file()
            and (directory / "assets" / "clean_repository.ipynb").is_file()
        ):
            continue

        try:
            project = tomllib.loads(
                pyproject.read_text(encoding="utf-8")
            )
        except (OSError, ValueError) as error:
            raise RuntimeError(
                f"Invalid project metadata in {pyproject}."
            ) from error

        if project.get("project", {}).get("name") == "qachat":
            return directory

    raise RuntimeError(
        f"Could not find the QAChat repository root from {current}. "
        "Open this notebook within the source checkout."
    )


In [ ]:
repository_root = find_repository_root()
print(f"QAChat repository: {repository_root}")


## Cleanup policy

Only explicitly recognized disposable artifacts are candidates for removal.
The scan never enters user data, model caches, Git internals, virtual
environments, or benchmark workspaces. Tracked paths are never deleted, even
if they happen to match a temporary-file rule. AppleDouble `._*` files must
have the AppleDouble signature, to avoid deleting unrelated hidden files.

The exclusions in `.gitignore` govern what Git tracks; **they are not a
list of files that the cleaner is allowed to delete**. In particular, ignored
model weights and workspaces are always protected.


In [ ]:
TEMPORARY_DIRECTORY_NAMES = {
    "__pycache__",
    "__MACOSX",
    ".ipynb_checkpoints",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    ".coverage_cache",
}

TEMPORARY_FILE_NAMES = {
    ".DS_Store",
    ".coverage",
    "Thumbs.db",
}

TEMPORARY_FILE_SUFFIXES = {".pyc", ".pyo"}

PROTECTED_DIRECTORY_NAMES = {
    ".git",
    ".venv",
    "venv",
    "env",
    ".models",
    ".weights",
    ".genrag",
    "sandbox",
    ".tox",
    ".nox",
}

PROTECTED_RELATIVE_DIRECTORIES = {
    ("benchmarks", "sandbox"),
    ("benchmarks", "results"),
}

APPLEDOUBLE_MAGIC = bytes.fromhex("00051607")


In [ ]:
def git_tracked_paths(root: Path) -> set[str] | None:
    """Return Git-tracked paths, or None outside a Git checkout.

    Paths are repository-relative POSIX strings. An invalid Git checkout
    raises an error instead of allowing an unsafe fallback.

    Parameters
    ----------
    root : Path
        Resolved QAChat repository root.

    Returns
    -------
    set[str] | None
        Tracked relative paths, or None when Git metadata is absent.

    """

    if not (root / ".git").exists():
        return None

    try:
        result = subprocess.run(
            ["git", "-C", str(root), "rev-parse", "--show-toplevel"],
            capture_output=True,
            check=True,
            text=True,
        )
        git_root = Path(result.stdout.strip()).resolve()
        if git_root != root.resolve():
            raise RuntimeError(
                f"Git root {git_root} differs from QAChat root {root}."
            )

        files = subprocess.run(
            ["git", "-C", str(root), "ls-files", "--cached", "-z"],
            capture_output=True,
            check=True,
        )
    except (OSError, subprocess.CalledProcessError) as error:
        raise RuntimeError("Could not read the Git file inventory.") from error

    return {
        os.fsdecode(item).replace(os.sep, "/")
        for item in files.stdout.split(b"\x00")
        if item
    }


def is_protected(path: Path, root: Path) -> bool:
    """Return whether a path belongs to protected repository data.

    Parameters
    ----------
    path : Path
        Candidate path.
    root : Path
        Resolved repository root.

    Returns
    -------
    bool
        True when any component or protected prefix must be preserved.

    """

    parts = path.relative_to(root).parts
    return (
        any(part in PROTECTED_DIRECTORY_NAMES for part in parts)
        or any(
            parts[:len(prefix)] == prefix
            for prefix in PROTECTED_RELATIVE_DIRECTORIES
        )
    )


def is_appledouble(path: Path) -> bool:
    """Identify a real AppleDouble companion file by its header.

    Parameters
    ----------
    path : Path
        Candidate file beginning with ``._``.

    Returns
    -------
    bool
        Whether it contains the AppleDouble file signature.

    """

    if not path.name.startswith("._") or path.is_symlink():
        return False
    try:
        with path.open("rb") as file:
            return file.read(4) == APPLEDOUBLE_MAGIC
    except OSError:
        return False


## Preview removable paths

The scanner avoids following symbolic links and skips directories under
protected locations. Candidates tracked by Git are excluded. Review these
lists before changing `DELETE` to `True`.


In [ ]:
def find_temporary_paths(
    root: Path,
) -> tuple[list[Path], list[Path]]:
    """Identify safe-to-remove generated paths below the checkout.

    Parameters
    ----------
    root : Path
        Resolved QAChat repository root.

    Returns
    -------
    tuple[list[Path], list[Path]]
        Candidate directories and files, sorted by relative path.

    """

    tracked = git_tracked_paths(root) or set()
    directories: list[Path] = []
    files: list[Path] = []

    for parent, names, filenames in os.walk(
        root, topdown=True, followlinks=False
    ):
        directory = Path(parent)
        retained_dirs = []

        for name in sorted(names):
            path = directory / name
            if path.is_symlink() or is_protected(path, root):
                continue

            relative = path.relative_to(root).as_posix()
            contains_tracked = any(
                item == relative or item.startswith(relative + "/")
                for item in tracked
            )
            if name in TEMPORARY_DIRECTORY_NAMES:
                if not contains_tracked:
                    directories.append(path)
                continue

            retained_dirs.append(name)

        names[:] = retained_dirs
        for name in sorted(filenames):
            path = directory / name
            if path.is_symlink() or is_protected(path, root):
                continue
            if path.relative_to(root).as_posix() in tracked:
                continue
            if (
                name in TEMPORARY_FILE_NAMES
                or path.suffix in TEMPORARY_FILE_SUFFIXES
                or is_appledouble(path)
            ):
                files.append(path)

    directories.sort(key=lambda path: path.relative_to(root).as_posix())
    files.sort(key=lambda path: path.relative_to(root).as_posix())
    return directories, files


In [ ]:
directories, files = find_temporary_paths(repository_root)

print("Removable directories:")
for path in directories:
    print("  ", path.relative_to(repository_root))
print("Removable files:")
for path in files:
    print("  ", path.relative_to(repository_root))
print(f"Preview: {len(directories)} directories, {len(files)} files.")


## Optional removal

**Default: preview only.** Set `DELETE = True` only after inspecting the
candidate lists. A fresh scan is performed immediately before deletion;
protected and Git-tracked paths remain excluded. No export operation requires
running the deletion step.


In [ ]:
DELETE = False  # Explicitly opt in after reviewing the preview.


In [ ]:
if not DELETE:
    print("No files deleted. Set DELETE = True to remove candidates.")
else:
    # Re-scan instead of trusting an earlier, potentially stale preview.
    directories, files = find_temporary_paths(repository_root)
    removed_directories = 0
    removed_files = 0

    for path in files:
        if (
            not path.is_symlink()
            and path.is_file()
            and path.resolve().is_relative_to(repository_root)
        ):
            path.unlink()
            removed_files += 1

    for path in directories:
        if (
            not path.is_symlink()
            and path.is_dir()
            and path.resolve().is_relative_to(repository_root)
        ):
            shutil.rmtree(path)
            removed_directories += 1

    print(
        f"Removed {removed_directories} directories "
        f"and {removed_files} files."
    )


In [ ]:
remaining_directories, remaining_files = find_temporary_paths(
    repository_root
)
if remaining_directories or remaining_files:
    print(
        f"Remaining candidates: {len(remaining_directories)} directories, "
        f"{len(remaining_files)} files."
    )
else:
    print("No removable temporary paths remain.")


## Export a clean source ZIP

The export uses Git's tracked-file inventory but copies **working-tree
contents**, so modifications to already tracked files are included. Untracked
files are deliberately omitted and reported. `.gitignore` is added explicitly
even if it has not yet been tracked.

The archive includes QAChat's normal source folders (`qachat/`, `tests/`,
`docs/`, `assets/`, `tutorials/`, and `setup/` where present), root project
files, `.gitignore`, and an explicitly **empty** `sandbox/` entry. It never
includes `sandbox/.gitkeep`, any other sandbox content, macOS metadata,
Git internals, local environments, model caches, or benchmark workspaces.

Git metadata must be available for a trustworthy file inventory. The export
refuses missing tracked files or symlinks rather than silently following them.
It also refuses to overwrite an existing destination ZIP.


In [ ]:
EXPORT_SKIPPED_DIRECTORY_NAMES = (
    PROTECTED_DIRECTORY_NAMES | TEMPORARY_DIRECTORY_NAMES
)

EXPORT_SKIPPED_FILE_NAMES = TEMPORARY_FILE_NAMES | {
    "Thumbs.db",
}


def should_export(relative: str) -> bool:
    """Apply explicit safety exclusions to a tracked source path.

    Parameters
    ----------
    relative : str
        Repository-relative path using POSIX separators.

    Returns
    -------
    bool
        Whether the path is safe to include in the exported source ZIP.

    """

    parts = tuple(relative.split("/"))
    if not parts or relative.startswith("/") or ".." in parts:
        return False
    if any(part in EXPORT_SKIPPED_DIRECTORY_NAMES for part in parts):
        return False
    if any(
        parts[:len(prefix)] == prefix
        for prefix in PROTECTED_RELATIVE_DIRECTORIES
    ):
        return False
    filename = parts[-1]
    if filename.startswith("._"):
        return False
    if filename in EXPORT_SKIPPED_FILE_NAMES:
        return False
    if filename in {".env", ".coverage"}:
        return False
    if filename.startswith(".env.") and filename != ".env.example":
        return False
    if Path(filename).suffix in TEMPORARY_FILE_SUFFIXES:
        return False
    return True


def list_source_files(root: Path) -> tuple[list[str], list[str]]:
    """List archive inputs and report excluded untracked files.

    Parameters
    ----------
    root : Path
        Resolved QAChat checkout root.

    Returns
    -------
    tuple[list[str], list[str]]
        Sorted source paths and sorted untracked, nonignored paths.

    Raises
    ------
    RuntimeError
        If the checkout has no Git inventory or a source path is unsafe.

    """

    tracked = git_tracked_paths(root)
    if tracked is None:
        raise RuntimeError(
            "ZIP export requires a Git checkout; .git is missing."
        )

    gitignore = root / ".gitignore"
    if not gitignore.is_file() or gitignore.is_symlink():
        raise RuntimeError("A regular .gitignore file is required.")

    selected = {item for item in tracked if should_export(item)}
    selected.add(".gitignore")
    for relative in selected:
        source = root / relative
        if (
            not source.is_file()
            or source.is_symlink()
            or not source.resolve().is_relative_to(root)
        ):
            raise RuntimeError(
                f"Missing, symbolic, or unsafe tracked file: {relative}"
            )

    try:
        untracked_output = subprocess.run(
            ["git", "-C", str(root), "ls-files", "--others",
             "--exclude-standard", "-z"],
            capture_output=True,
            check=True,
        ).stdout
    except (OSError, subprocess.CalledProcessError) as error:
        raise RuntimeError("Could not inspect untracked files.") from error

    untracked = sorted(
        os.fsdecode(item).replace(os.sep, "/")
        for item in untracked_output.split(b"\x00")
        if item and os.fsdecode(item) != ".gitignore"
    )
    return sorted(selected), untracked


In [ ]:
source_files, untracked_files = list_source_files(repository_root)

print(f"ZIP preview: {len(source_files)} files, plus empty sandbox/.")
print("Included top-level entries:")
for item in sorted({path.split("/")[0] for path in source_files}):
    print("  ", item)
print("   sandbox/ (empty)")
if untracked_files:
    print(f"WARNING: {len(untracked_files)} untracked files excluded:")
    for item in untracked_files:
        print("  ", item)
print("Verify new source files are tracked before exporting.")


In [ ]:
def export_source_zip(
    root: Path,
    destination: Path,
    source_files: list[str],
) -> Path:
    """Write and verify an archive without modifying the source tree.

    The archive is created in a temporary file in the destination folder.
    Files are streamed from the working tree, and every directory entry has
    a fixed timestamp for reproducible exports.

    Parameters
    ----------
    root : Path
        Resolved QAChat checkout root.
    destination : Path
        Destination ZIP path, outside the checkout.
    source_files : list[str]
        Validated relative file paths from :func:`list_source_files`.

    Returns
    -------
    Path
        Path to the verified ZIP file.

    Raises
    ------
    RuntimeError
        If the ZIP would include an unsafe source or fails verification.
    FileExistsError
        If the requested archive already exists.

    """

    root = root.resolve()
    destination = destination.resolve()
    if destination.is_relative_to(root):
        raise RuntimeError("Export the archive outside the checkout.")
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite {destination}.")
    if not destination.parent.is_dir():
        raise RuntimeError("The ZIP destination directory does not exist.")

    expected = set(source_files)
    expected.add(".gitignore")
    if not all(should_export(item) for item in expected):
        raise RuntimeError("The source list includes a protected path.")
    if any(item.startswith("sandbox/") for item in expected):
        raise RuntimeError("The sandbox must be empty in the export.")

    directories = {"sandbox/"}
    for relative in expected:
        parent = Path(relative).parent
        while parent != Path("."):
            directories.add(parent.as_posix() + "/")
            parent = parent.parent

    timestamp = (1980, 1, 1, 0, 0, 0)
    with tempfile.NamedTemporaryFile(
        prefix=".qachat-export-",
        suffix=".zip.tmp",
        dir=destination.parent,
        delete=False,
    ) as temporary:
        temp_path = Path(temporary.name)

    try:
        with zipfile.ZipFile(temp_path, "w") as archive:
            for relative in sorted(directories):
                info = zipfile.ZipInfo(relative, timestamp)
                info.create_system = 3
                info.external_attr = (stat.S_IFDIR | 0o755) << 16
                info.external_attr |= 0x10
                archive.writestr(info, b"")

            for relative in sorted(expected):
                source = root / relative
                if (
                    not source.is_file()
                    or source.is_symlink()
                    or not source.resolve().is_relative_to(root)
                ):
                    raise RuntimeError(f"Unsafe source: {relative}")
                info = zipfile.ZipInfo(relative, timestamp)
                info.create_system = 3
                info.external_attr = (
                    stat.S_IFREG | (source.stat().st_mode & 0o777)
                ) << 16
                info.compress_type = zipfile.ZIP_DEFLATED
                with source.open("rb") as input_file:
                    with archive.open(info, "w") as output_file:
                        shutil.copyfileobj(input_file, output_file)

        with zipfile.ZipFile(temp_path) as archive:
            names = archive.namelist()
            if (
                len(names) != len(set(names))
                or set(names) != expected | directories
                or archive.testzip() is not None
                or any(
                    name.startswith("sandbox/") and name != "sandbox/"
                    for name in names
                )
            ):
                raise RuntimeError("ZIP integrity or content check failed.")

        temp_path.replace(destination)
    finally:
        temp_path.unlink(missing_ok=True)

    return destination


## Optional ZIP creation

The destination defaults to the **parent of the repository**, so it cannot
be accidentally included in the source tree. Change the filename if an
archive already exists: overwriting an existing ZIP is intentionally refused.

Set `EXPORT_ZIP = True` to create the archive after inspecting the preview.
You do not need to turn on cleanup to export a clean archive.


In [ ]:
EXPORT_ZIP = False  # Explicitly opt in after previewing the inputs.
ZIP_PATH = repository_root.parent / (
    f"{repository_root.name}_source.zip"
)


In [ ]:
if not EXPORT_ZIP:
    print(f"ZIP preview only. Set EXPORT_ZIP = True for {ZIP_PATH}.")
else:
    # Recompute inputs so a stale preview cannot select obsolete files.
    source_files, untracked_files = list_source_files(repository_root)
    if untracked_files:
        print(
            f"NOTE: {len(untracked_files)} untracked files are omitted."
        )
    archive_path = export_source_zip(
        repository_root, ZIP_PATH, source_files
    )
    print(f"Verified clean source archive: {archive_path}")
